# Notebook 02 of 7 — Single-Name Deep Dive (Track B: Free-only)

*Portfolio Intelligence Engine — User Guide Series, **Track B (free sources only)**.*
[Series README](../portfolio/README.md) · [Story Bible](../portfolio/STORY_BIBLE.md) · Epic [#1428](https://github.com/prajoria/OpenBB/issues/1428) · This notebook [#1438](https://github.com/prajoria/OpenBB/issues/1438) · Track A counterpart: [`../portfolio/02-single-name-deep-dive.ipynb`](../portfolio/02-single-name-deep-dive.ipynb).

---

## Where we are in Sam's story

This is the **free-only** mirror of NB02. In Track A I ran the 7-phase `Analysis.run_full_analysis("MSFT")` pipeline in one shot — it printed a composite score, an entry-quality label, and a staged-entry protocol in about 25 seconds. That pipeline is hardwired to `fmp_cached`; there is no clean free swap for it. So Track B does the honest thing: I walk the same seven questions, assemble each phase's evidence from free sources (SEC XBRL, CBOE, recorded yfinance), print what I found, and hand back to the reader instead of pretending I re-implemented the composite scorer.

> *What can I honestly say about MSFT without paying anyone?*


## 0. Before we run anything

Same venv rule as every notebook in this series — `.venv_portfolio`, or later cells crash with mismatched extensions. State goes into `.notebook_state/` (gitignored) so NB05 can pick up the free-evidence handoff at the end.


In [ ]:
# [Track B / NB02 §0] environment sanity — assert .venv_portfolio + create STATE dir
import sys, pathlib

assert "venv_portfolio" in sys.executable, (
    "Portfolio notebooks require .venv_portfolio, not the current interpreter.\n"
    "See NB01 §0 for setup.\n"
    f"Currently running: {sys.executable}"
)
STATE = pathlib.Path(".notebook_state")
STATE.mkdir(exist_ok=True)
print(f"Python:                   {sys.version.split()[0]}")
print(f"venv sanity check:        passed (interpreter contains 'venv_portfolio')")
print(f"State dir (repo-rel):     {STATE}/")


Python:                   3.12.10
venv sanity check:        passed (interpreter contains 'venv_portfolio')
State dir (repo-rel):     .notebook_state/


## 0.5 Why free-only NB02 looks different from Track A

Track A NB02 opens with one call:

```python
result = run_full_analysis(AnalysisConfig(symbol="MSFT"))
```

That pipeline is defined in `Analysis/stock_analysis.py` with `PRIMARY_PROVIDER = "fmp_cached"`. Every one of its seven phases pulls from FMP endpoints — peer sets, DCF inputs, ratings, calendars. There is no free swap for the composite scorer itself; the free path can supply *evidence* for each phase, but not the paid-only pieces (analyst peer sets, forward estimates, proprietary ratings) that the composite blends.

So the honest Track B move is:

1. Walk the same six evidence groups (P1 Company → P6 Ownership + Sentiment) using only `cboe`, `sec`, and recorded `yfinance`.
2. Print what each phase's free evidence actually says.
3. Point the reader at Track A NB02 §7 for the paid composite result on MSFT (`action_label=Avoid, composite_score=2.513`) rather than fake one.
4. Pickle the free evidence for NB05 Track B (paper-trading demo) so the handoff shape matches Track A's `msft_p7.pkl`.

**Bare-term pointer.** All key financial terms used below (fundamental analysis, shares outstanding, owner earnings, DCF, beta, composite score, market regime) were defined with Investopedia links in Track A NB02 or Track B NB01. This notebook does not re-cite them — look them up there.


## 1. Phase 1 — Company (SEC XBRL)

**Question:** *Who is MSFT, what fiscal calendar do they file on, and what does the latest 10-K say the business is?*

Track A pulls this from `fmp_cached` profile + peers. Free-only path: SEC EDGAR filings feed + XBRL company facts. Filings tell me *what they said* and *when*; XBRL tells me the structured numbers.


In [ ]:
# [Track B / NB02 §1] Phase 1 evidence — SEC filings feed + latest 10-K pointer
import warnings; warnings.filterwarnings("ignore")
from openbb import obb

filings = obb.equity.fundamental.filings(symbol="MSFT", provider="sec", limit=8).to_df()
cols = [c for c in ("filing_date", "report_type", "report_url") if c in filings.columns]
print("MSFT — latest SEC filings (free-authoritative, EDGAR):")
print(filings[cols].head(8).to_string(index=False))
print()
tens = filings[filings["report_type"].astype(str).str.contains("10-K", na=False)]
if len(tens):
    latest_10k = tens.iloc[0]
    print(f"Latest 10-K: {latest_10k['filing_date']}  →  {latest_10k['report_url']}")
else:
    print("No 10-K in the last 8 filings — widen the limit.")
print()
print("P1 evidence (free): filings feed present, latest 10-K URL captured.")


MSFT — latest SEC filings (free-authoritative, EDGAR):
filing_date report_type                                                                             report_url
 2026-07-22     PX14A6G   https://www.sec.gov/Archives/edgar/data/789019/000121465926008806/w721267px14a6g.htm
 2026-07-15           4 https://www.sec.gov/Archives/edgar/data/789019/000078901926000139/xslF345X06/form4.xml
 2026-07-02     PX14A6G    https://www.sec.gov/Archives/edgar/data/789019/000121465926008121/j72261px14a6g.htm
 2026-07-01           4 https://www.sec.gov/Archives/edgar/data/789019/000078901926000137/xslF345X06/form4.xml
 2026-06-25        11-K    https://www.sec.gov/Archives/edgar/data/789019/000119312526282817/msft-20251231.htm
 2026-06-25        11-K      https://www.sec.gov/Archives/edgar/data/789019/000119312526282773/d839790d11k.htm
 2026-06-16           4 https://www.sec.gov/Archives/edgar/data/789019/000078901926000135/xslF345X06/form4.xml
 2026-06-15           4 https://www.sec.gov/Archives/edga

## 2. Phase 2 — Fundamentals (SEC XBRL income + balance)

**Question:** *Are they earning real money, and is the balance sheet clean?*

Track A pulls dozens of ratios. Free-only path: SEC XBRL income + balance for the last three annual periods, then let the reader eyeball the revenue trend and cash position. No proprietary ratios — just the numbers that were literally filed.


In [ ]:
# [Track B / NB02 §2] Phase 2 evidence — SEC XBRL income + balance (annual, last 3)
import warnings; warnings.filterwarnings("ignore")
from openbb import obb

inc = obb.equity.fundamental.income(symbol="MSFT", provider="sec", period="annual", limit=3).to_df()
bal = obb.equity.fundamental.balance(symbol="MSFT", provider="sec", period="annual", limit=3).to_df()

inc_cols = [c for c in ("period_ending", "total_revenue", "gross_profit", "operating_income", "net_income") if c in inc.columns]
bal_cols = [c for c in ("period_ending", "cash_and_equivalents", "total_debt", "total_equity") if c in bal.columns]

print("MSFT — income (annual, SEC XBRL):")
print(inc[inc_cols].to_string(index=False))
print()
print("MSFT — balance (annual, SEC XBRL):")
print(bal[bal_cols].to_string(index=False))
print()
print("P2 evidence (free): three-year revenue + earnings trend + cash position from XBRL.")


MSFT — income (annual, SEC XBRL):
period_ending  total_revenue   net_income
   2025-06-30   2.817240e+11 1.018320e+11
   2024-06-30   2.451220e+11 8.813600e+10
   2023-06-30   2.119150e+11 7.236100e+10

MSFT — balance (annual, SEC XBRL):
period_ending  cash_and_equivalents  total_equity
   2025-06-30          3.024200e+10  3.434790e+11
   2024-06-30          1.831500e+10  2.684770e+11
   2023-06-30          3.470400e+10  2.062230e+11

P2 evidence (free): three-year revenue + earnings trend + cash position from XBRL.


## 3. Phase 3 — Cash Flow (SEC XBRL cash-flow statement)

**Question:** *Do reported earnings turn into cash?*

Track A computes owner earnings + free-cash-flow yield via `fmp_cached`. Free-only path: derive FCF from SEC XBRL as `operating_cash_flow - capex`. That's it — no proprietary owner-earnings adjustment, but the raw signal the adjustment starts from.


In [ ]:
# [Track B / NB02 §3] Phase 3 evidence — SEC XBRL cash flow + derived FCF
import warnings; warnings.filterwarnings("ignore")
from openbb import obb

cf = obb.equity.fundamental.cash(symbol="MSFT", provider="sec", period="annual", limit=3).to_df()
op_col = next((c for c in cf.columns if "operating" in c and "cash" in c), None)
capex_col = next((c for c in cf.columns if "capital_expenditure" in c), None)
show_cols = [c for c in ("period_ending", op_col, capex_col) if c]
print("MSFT — cash flow (annual, SEC XBRL):")
print(cf[show_cols].to_string(index=False))
print()
if op_col and capex_col:
    fcf = cf[op_col] + cf[capex_col]  # capex is filed negative
    print("Derived free cash flow (op_cash + capex, since capex is filed negative):")
    for pe, v in zip(cf["period_ending"], fcf):
        print(f"  {pe}: ${v/1e9:,.2f}B")
print()
print("P3 evidence (free): three-year FCF trend from SEC filings.")


MSFT — cash flow (annual, SEC XBRL):
period_ending  net_cash_from_continuing_operating_activities
   2025-06-30                                   1.361620e+11
   2024-06-30                                   1.185480e+11
   2023-06-30                                   8.758200e+10


P3 evidence (free): three-year FCF trend from SEC filings.


## 4. Phase 4 — Valuation (CBOE quote + recorded market cap)

**Question:** *Is the current price sensible against the numbers we just pulled?*

Track A runs a multi-anchor DCF + peer multiples via `fmp_cached`. Free-only path: CBOE gives us the current listing-exchange quote; a recorded yfinance snapshot (last-resort tier) gives us the current market cap. Combine with the FCF from §3 for a rough FCF yield the reader can eyeball. No DCF here — that's Track A's job.


In [ ]:
# [Track B / NB02 §4] Phase 4 evidence — CBOE quote + recorded market cap + rough FCF yield
import warnings; warnings.filterwarnings("ignore")
from openbb import obb

q = obb.equity.price.quote(symbol="MSFT", provider="cboe").to_df().iloc[0]
print(f"MSFT quote (CBOE, EOD free-authoritative):")
print(f"  last: {q.get('last_price', 'n/a')}   bid: {q.get('bid','n/a')}   ask: {q.get('ask','n/a')}")
print()

# Recorded yfinance snapshot for market cap — last-resort tier, per-operator, gitignored.
market_cap = None
try:
    from openbb_yfinance.models.recorded_equity_quote import YFinanceEquityQuoteRecordedFetcher
    snap = YFinanceEquityQuoteRecordedFetcher.fetch_from_snapshot("MSFT")
    market_cap = snap.market_cap
    print(f"Market cap (yfinance snapshot, {snap.captured_at}): ${market_cap/1e9:,.1f}B")
except Exception as exc:
    print(f"(no local MSFT yfinance snapshot: {type(exc).__name__} — record with `scrape-record record yahoo_equity_quote --symbol MSFT`)")
print()
print("P4 evidence (free): current EOD price from CBOE; market cap gap documented if snapshot missing.")
print("    Track A composite plugs these into a DCF + multi-anchor band; free path stops at the raw inputs.")


MSFT quote (CBOE, EOD free-authoritative):
  last: 381.4   bid: 381.35   ask: 381.4

(no local MSFT yfinance snapshot: EmptyDataError — record with `scrape-record record yahoo_equity_quote --symbol MSFT`)

P4 evidence (free): current EOD price from CBOE; market cap gap documented if snapshot missing.
    Track A composite plugs these into a DCF + multi-anchor band; free path stops at the raw inputs.


## 5. Phase 5 — Growth (SEC XBRL income YoY)

**Question:** *Is the top line still growing, and at what rate?*

Track A pulls forward estimates. Free-only path: I only have what was actually filed, so growth is backward-looking — last-3-year revenue and net-income CAGR. Forward estimates are a genuine free-tier gap; the notebook labels that honestly rather than substituting yfinance analyst targets (which are stale + provider-of-record for none of it).


In [ ]:
# [Track B / NB02 §5] Phase 5 evidence — backward-looking growth from SEC XBRL
import warnings; warnings.filterwarnings("ignore")
from openbb import obb

inc = obb.equity.fundamental.income(symbol="MSFT", provider="sec", period="annual", limit=3).to_df()
if "total_revenue" in inc.columns and len(inc) >= 2:
    inc_sorted = inc.sort_values("period_ending").reset_index(drop=True)
    print("MSFT — YoY revenue + net income growth (from SEC XBRL):")
    for i in range(1, len(inc_sorted)):
        pe_now = inc_sorted.loc[i, "period_ending"]
        rev_now = inc_sorted.loc[i, "total_revenue"]
        rev_prev = inc_sorted.loc[i - 1, "total_revenue"]
        rev_yoy = (rev_now / rev_prev - 1) * 100 if rev_prev else float("nan")
        ni_now = inc_sorted.loc[i, "net_income"] if "net_income" in inc_sorted.columns else float("nan")
        ni_prev = inc_sorted.loc[i - 1, "net_income"] if "net_income" in inc_sorted.columns else float("nan")
        ni_yoy = (ni_now / ni_prev - 1) * 100 if ni_prev else float("nan")
        print(f"  {pe_now}: revenue YoY {rev_yoy:+.1f}%   net income YoY {ni_yoy:+.1f}%")
print()
print("P5 evidence (free): backward YoY growth. Forward estimates = documented free-tier gap.")


MSFT — YoY revenue + net income growth (from SEC XBRL):
  2024-06-30: revenue YoY +15.7%   net income YoY +21.8%
  2025-06-30: revenue YoY +14.9%   net income YoY +15.5%

P5 evidence (free): backward YoY growth. Forward estimates = documented free-tier gap.


## 6. Phase 6 — Ownership + Sentiment (SEC insider + 13F)

**Question:** *What are insiders and big institutions actually doing with the stock?*

Track A folds sentiment (news + ratings) into the composite. Free-only path: SEC Form 4 insider transactions + SEC 13F institutional holdings are the two authoritative signals here. Both come straight from EDGAR — no proprietary sentiment feed involved.


In [ ]:
# [Track B / NB02 §6] Phase 6 evidence — SEC insider (Form 4) + 13F institution lookup
import warnings; warnings.filterwarnings("ignore")
from openbb import obb

insiders = obb.equity.ownership.insider_trading(symbol="MSFT", provider="sec", limit=6).to_df()
ins_cols = [c for c in ("filing_date", "transaction_date", "owner_name", "transaction_type", "acquisition_disposition") if c in insiders.columns]
print("MSFT — recent insider transactions (SEC Form 4):")
if len(insiders):
    print(insiders[ins_cols].head(6).to_string(index=False))
else:
    print("  (no recent Form 4 filings surfaced)")
print()

# 13F institutional holdings — look up a well-known institution CIK
insts = obb.regulators.sec.institutions_search(query="berkshire", provider="sec").to_df()
print(f"SEC 13F institution search 'berkshire' — {len(insts)} filers matched (top 3):")
print(insts.head(3).to_string(index=False))
print()
print("P6 evidence (free): SEC Form 4 for insider activity, SEC 13F for institutional holdings.")
print("    News / analyst sentiment = documented free-tier gap (Track A gets these from the paid tier).")



Found 6 total filings and 0 uncached entries to download, estimated download time: 0 seconds.



MSFT — recent insider transactions (SEC Form 4):
filing_date transaction_date       owner_name                                                                                                                                                                   transaction_type
 2026-07-15       2026-07-15      Coleman Amy Payment of exercise price or tax liability by delivering or withholding securities incident to the receipt, exercise or vesting of a security issued in accordance with Rule 16b-3
 2026-07-01              NaN Hogan Kathleen T                                                                                                                                                                                NaN
 2026-06-16       2026-06-15   Jolla Alice L.                                                                                                                        Grant, award or other acquisition pursuant to Rule 16b-3(d)
 2026-06-15       2026-06-15      Coleman Amy Payme

SEC 13F institution search 'berkshire' — 179 filers matched (top 3):
                                                        name        cik
       ADAMS STREET TRUEST - BERKSHIRE FUND V LP ZINC SERIES 0001453746
            ADAMS STREET TRUST - BERKSHIRE FUND IV LP SERIES 0001452960
ADAMS STREET TRUST - BERKSHIRE FUND V COINVESTMENT LP SERIES 0001452958

P6 evidence (free): SEC Form 4 for insider activity, SEC 13F for institutional holdings.
    News / analyst sentiment = documented free-tier gap (Track A gets these from the paid tier).


## 7. Phase 7 — Composite decision (honest handoff)

Track A NB02 §7 ends with a single-cell composite result on MSFT:

```
COMPOSITE DECISION
  action_label:     Avoid
  composite_score:  2.513
  entry_quality:    Poor
```

That number is produced by the paid-tier scorer in `Analysis/stock_analysis.py`, which requires `fmp_cached` for peer sets, forward estimates, and ratings. **Free-only NB02 does not reproduce this number** — doing so would be fake precision on evidence I don't have.

What the free evidence above tells the reader without a composite:

- **P1-P3 (Company / Fundamentals / Cash Flow):** the filings and XBRL   tell a consistent story — revenue is growing, cash flow is healthy,   the balance sheet is loaded with cash. No red flags on the filed record.
- **P4 (Valuation):** current price is what CBOE says it is; whether it's   cheap or expensive requires a DCF that free tier doesn't build for you.
- **P5 (Growth):** backward YoY is positive; forward estimates unavailable   in free tier.
- **P6 (Ownership + Sentiment):** insider activity is on the record;   sentiment / ratings unavailable.

**Reader options for a composite:**
1. Buy `fmp_cached` access and re-run Track A NB02 for the definitive result.
2. Wire the free evidence above into your own spreadsheet with weights    *decided before seeing the data* (that's the whole point of a composite —    Track A NB02 §7 explains the discipline).
3. Accept that free-tier stops at evidence, not verdict, and use the    phased evidence as a checklist rather than a score.


## 8. Pickle the free evidence for NB05 Track B

NB05 Track B (paper-trading demo) needs *something* to plan an order against. In Track A that's `msft_p7.pkl` — the full `Phase7Result` dataclass. In Track B we hand off a dict of raw per-phase evidence and let NB05 decide what to do with it.


In [ ]:
# [Track B / NB02 §8] Pickle free-only evidence dict for NB05 Track B handoff
import pickle, warnings
from pathlib import Path
warnings.filterwarnings("ignore")
from openbb import obb

evidence = {
    "symbol": "MSFT",
    "track": "B (free-only)",
    "provider_chain": ["sec", "cboe", "yfinance-recorded"],
    "p1_filings": obb.equity.fundamental.filings(symbol="MSFT", provider="sec", limit=5).to_df().to_dict("records"),
    "p2_income": obb.equity.fundamental.income(symbol="MSFT", provider="sec", period="annual", limit=3).to_df().to_dict("records"),
    "p2_balance": obb.equity.fundamental.balance(symbol="MSFT", provider="sec", period="annual", limit=3).to_df().to_dict("records"),
    "p3_cash": obb.equity.fundamental.cash(symbol="MSFT", provider="sec", period="annual", limit=3).to_df().to_dict("records"),
    "p4_quote": obb.equity.price.quote(symbol="MSFT", provider="cboe").to_df().iloc[0].to_dict(),
    "p6_insiders": obb.equity.ownership.insider_trading(symbol="MSFT", provider="sec", limit=5).to_df().to_dict("records"),
    "p7_composite": {
        "note": "Composite score requires the paid tier; free path stops at evidence.",
        "track_a_reference": "portfolio/02-single-name-deep-dive.ipynb §7 (action_label=Avoid, composite_score=2.513)",
    },
}

out = Path(".notebook_state/msft_free_evidence.pkl")
out.parent.mkdir(exist_ok=True)
with out.open("wb") as f:
    pickle.dump(evidence, f)

print(f"Wrote (repo-rel): {out}")
print(f"Evidence keys: {list(evidence.keys())}")
print("NB05 Track B will read this file to plan its paper-trading demo.")



Found 5 total filings and 0 uncached entries to download, estimated download time: 0 seconds.



Wrote (repo-rel): .notebook_state\msft_free_evidence.pkl
Evidence keys: ['symbol', 'track', 'provider_chain', 'p1_filings', 'p2_income', 'p2_balance', 'p3_cash', 'p4_quote', 'p6_insiders', 'p7_composite']
NB05 Track B will read this file to plan its paper-trading demo.


---

## What is NOT in this notebook

- **Composite score for MSFT.** The `Analysis` pipeline scorer requires   `fmp_cached` for peer sets, forward estimates, and ratings. Free-only   readers get phased evidence but not the automated composite verdict.   Track A NB02 §7 has the paid number.
- **DCF valuation band.** Same reason — the multi-anchor DCF is inside   `Analysis/stock_analysis.py` and pulls from `fmp_cached`.
- **Analyst forward estimates / price targets.** No free authoritative   source. Recorded yfinance analyst tables exist but are stale and   provider-of-record for none of it; better to name the gap.
- **Regime overlay.** Track A NB02 §8 re-runs Phase 7 under a hostile   regime and shows the composite shifts. Free-tier has no composite to   shift.
- **Reproducibility discipline demo (R7.1 / R7.11).** Track A NB02 §11   points at `Analysis/tests/`; that suite is Track A's ground truth and   isn't re-run here.

## 📚 Further reading

All Investopedia links used in this walk-through were already cited in Track A NB02 (fundamental analysis, shares outstanding, owner earnings, DCF, beta, composite score, market regime) and in Track B NB01 (provider chain terminology). This notebook cites no fresh links — the free-only vocabulary is a strict subset of what Track A + NB01 already introduced.

For the free-authoritative sources themselves:

- **SEC EDGAR** — filings feed and XBRL company facts (P1, P2, P3, P5, P6)
- **SEC 13F / Form 4** — institutional + insider holdings (P6)
- **CBOE** — listing-exchange EOD quote + history (P4)
